# 01 — Exploração Inicial dos Dados
**Olist Analytics Hub | Etapa 1 de 7**

**Objetivo:** Entender o schema relacional, inspecionar a qualidade dos dados e formular as perguntas de negócio que guiarão toda a análise.

---
### Perguntas que este notebook responde
1. Quantos registros e colunas tem cada dataset?
2. Como as tabelas se relacionam entre si?
3. Qual a qualidade dos dados (nulos, duplicatas, tipos)?
4. Quais são as distribuições básicas das variáveis-chave?

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, HTML

from src.data_loader import load_all_raw, validate_datasets, get_dataset_info
from src.visualizations import setup_style, save_fig

setup_style()
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

print('✓ Ambiente configurado.')

## 1. Carregamento dos Dados

In [ ]:
print('Carregando datasets...')
datasets = load_all_raw()

In [ ]:
# Visão geral de todos os datasets
info = get_dataset_info(datasets)
display(info.style.background_gradient(subset=['linhas'], cmap='Blues'))

## 2. Validação das Chaves Primárias

In [ ]:
print('Validando integridade dos datasets...')
validate_datasets(datasets)

## 3. Schema Relacional

```
┌─────────────────────┐       ┌──────────────────────┐
│     customers       │       │       sellers        │
│ PK: customer_id     │       │ PK: seller_id        │
└──────────┬──────────┘       └──────────┬───────────┘
           │                             │
           ▼                             ▼
┌─────────────────────────────────────────────────────┐
│                       orders                        │
│ PK: order_id  │  FK: customer_id                    │
└────────────────────────┬────────────────────────────┘
                         │
          ┌──────────────┼──────────────┬─────────────┐
          ▼              ▼              ▼             ▼
   ┌─────────────┐ ┌──────────┐ ┌──────────┐ ┌──────────────┐
   │ order_items │ │ payments │ │  reviews │ │  geolocation │
   │ FK:order_id │ │FK:ord_id │ │FK:ord_id │ │  (zip_code)  │
   │ FK:product_id│ └──────────┘ └──────────┘ └──────────────┘
   │ FK:seller_id│
   └──────┬──────┘
          ▼
   ┌─────────────┐
   │  products   │
   │PK:product_id│
   └─────────────┘
```

## 4. Inspeção Detalhada por Tabela

In [ ]:
# ── Orders ────────────────────────────────────────────────────────────────────
orders = datasets['orders']
print('=== ORDERS ===')
print(f'Shape: {orders.shape}')
print('\nTipos:')
print(orders.dtypes.to_string())
print('\nPrimeiras linhas:')
display(orders.head(3))

In [ ]:
# Distribuição do status dos pedidos
status_counts = orders['order_status'].value_counts()
print('Distribuição de Status:')
print(status_counts)
print(f'\nPedidos entregues: {status_counts.get("delivered", 0) / len(orders):.1%}')

In [ ]:
# Nulos por coluna em orders
null_analysis = orders.isnull().sum().rename('nulos')
null_analysis = null_analysis[null_analysis > 0].sort_values(ascending=False)
null_pct = (null_analysis / len(orders) * 100).rename('%')
display(pd.concat([null_analysis, null_pct], axis=1))

In [ ]:
# ── Order Items ───────────────────────────────────────────────────────────────
items = datasets['order_items']
print('=== ORDER ITEMS ===')
print(f'Shape: {items.shape}')
display(items.head(3))
print('\nEstatísticas de preço:')
display(items[['price', 'freight_value']].describe())

In [ ]:
# ── Customers ─────────────────────────────────────────────────────────────────
customers = datasets['customers']
print('=== CUSTOMERS ===')
print(f'Shape: {customers.shape}')
print(f'Clientes únicos: {customers["customer_unique_id"].nunique():,}')
print(f'Estados: {customers["customer_state"].nunique()}')
print('\nTop 10 estados por volume de clientes:')
print(customers['customer_state'].value_counts().head(10).to_string())

In [ ]:
# ── Products ──────────────────────────────────────────────────────────────────
products = datasets['products']
translation = datasets['product_category_translation']
print('=== PRODUCTS ===')
print(f'Shape: {products.shape}')
print(f'Categorias: {products["product_category_name"].nunique()}')
print('\nNulos por coluna:')
print(products.isnull().sum()[products.isnull().sum() > 0].to_string())

In [ ]:
# ── Reviews ───────────────────────────────────────────────────────────────────
reviews = datasets['order_reviews']
print('=== ORDER REVIEWS ===')
print(f'Shape: {reviews.shape}')
print(f'Pedidos com review: {reviews["order_id"].nunique():,}')
print('\nDistribuição de notas:')
print(reviews['review_score'].value_counts().sort_index().to_string())
print(f'\nNota média: {reviews["review_score"].mean():.2f}')

## 5. Visualizações Iniciais

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Status dos pedidos
status_data = orders['order_status'].value_counts()
axes[0].bar(status_data.index, status_data.values, color='#2E86AB', alpha=0.85)
axes[0].set_title('Status dos Pedidos', fontweight='bold')
axes[0].set_xlabel('Status')
axes[0].set_ylabel('Quantidade')
axes[0].tick_params(axis='x', rotation=45)

# 2. Distribuição de notas de avaliação
review_data = reviews['review_score'].value_counts().sort_index()
colors_review = ['#E94F37', '#F18F01', '#F18F01', '#44BBA4', '#2E86AB']
axes[1].bar(review_data.index, review_data.values, color=colors_review, alpha=0.9)
axes[1].set_title('Distribuição de Notas de Avaliação', fontweight='bold')
axes[1].set_xlabel('Nota (1–5)')
axes[1].set_ylabel('Quantidade')

# 3. Top 10 estados por clientes
state_data = customers['customer_state'].value_counts().head(10)
axes[2].barh(state_data.index, state_data.values, color='#A23B72', alpha=0.85)
axes[2].set_title('Top 10 Estados por Volume de Clientes', fontweight='bold')
axes[2].set_xlabel('Clientes')
axes[2].invert_yaxis()

plt.tight_layout()
save_fig('01_visao_geral')
plt.show()

In [ ]:
# Distribuição de preços com outliers
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(items['price'], bins=100, color='#2E86AB', alpha=0.8, edgecolor='white')
axes[0].set_title('Distribuição de Preços (todos os itens)', fontweight='bold')
axes[0].set_xlabel('Preço (R$)')
axes[0].set_ylabel('Frequência')
axes[0].axvline(items['price'].mean(), color='#E94F37', linestyle='--', linewidth=2, label=f'Média: R$ {items["price"].mean():.2f}')
axes[0].axvline(items['price'].median(), color='#F18F01', linestyle='--', linewidth=2, label=f'Mediana: R$ {items["price"].median():.2f}')
axes[0].legend()

# Boxplot sem outliers extremos (percentil 99)
p99 = items['price'].quantile(0.99)
axes[1].boxplot(items.loc[items['price'] <= p99, 'price'], vert=True)
axes[1].set_title(f'Boxplot de Preços (até p99 = R$ {p99:.0f})', fontweight='bold')
axes[1].set_ylabel('Preço (R$)')
axes[1].set_xticks([])

plt.tight_layout()
save_fig('01_distribuicao_precos')
plt.show()

## 6. Mapa de Nulos (Heat Map)

Visualização da completude de cada dataset para identificar quais tabelas precisarão de mais tratamento.

In [ ]:
def null_heatmap(df, title):
    null_pct = (df.isnull().mean() * 100).round(1)
    if null_pct.sum() == 0:
        print(f'  {title}: sem nulos.')
        return
    fig, ax = plt.subplots(figsize=(max(8, len(df.columns) * 0.6), 3))
    ax.bar(null_pct.index, null_pct.values, color=['#E94F37' if v > 5 else '#F18F01' if v > 0 else '#44BBA4' for v in null_pct.values])
    ax.set_title(f'% de Nulos por Coluna — {title}', fontweight='bold')
    ax.set_ylabel('%')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

for name, df in datasets.items():
    null_heatmap(df, name)

## 7. Sumário e Conclusões

| Aspecto | Achado |
|---------|--------|
| Volume total | ~100k pedidos, 9 tabelas |
| Pedidos entregues | ~97% do total |
| Clientes únicos | ~96k (baixa recompra) |
| Nota média | ~4.07/5 |
| Principal estado | SP (42% dos clientes) |
| Nulos críticos | Datas de entrega ausentes em pedidos cancelados/em trânsito |

### Decisões de limpeza para o próximo notebook:
1. Converter todas as colunas de data para `datetime`
2. Filtrar pedidos com status `delivered` para análises de entrega
3. Preencher nulos de peso/dimensão de produtos com mediana por categoria
4. Agregar pagamentos múltiplos por pedido (parcelamento)
5. Manter apenas 1 review por pedido (o mais recente)

---
**Próximo:** `02_limpeza_e_enriquecimento.ipynb`